In [ ]:

# =========================================================================
# 0. IMPORTS
# =========================================================================
import os, copy, time, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
import seaborn as sns
from PIL import Image
from collections import defaultdict
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

try:
    import timm; HAS_TIMM = True
except ImportError:
    HAS_TIMM = False; print("Tip: pip install timm to unlock ConvNeXt/Swin/RegNet")

from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, precision_recall_curve,
    average_precision_score, accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from scipy import stats

try:
    from skimage.metrics import structural_similarity as ssim; HAS_SSIM = True
except ImportError:
    HAS_SSIM = False


# =========================================================================
# 1. CONFIGURATION
# =========================================================================
CONFIG = {
    'result_dir':   './results',
    'out_dir':      './results/multi_model',
    'weights_dir':  './results/weights',
    'img_size':     224,
    'batch_size':   8,
    'num_classes':  2,
    'class_names':  ['Insensitive', 'Sensitive'],
    'num_workers':  0,
    'epochs':       10,
    'lr':           1e-4,
    'weight_decay': 1e-2,
    'seed':         42,
    'bootstrap_n':  1000,
    'dpi':          300,
    'device':       'cuda' if torch.cuda.is_available() else 'cpu',
}

for d in [CONFIG['out_dir'], CONFIG['weights_dir']]:
    os.makedirs(d, exist_ok=True)

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])


# =========================================================================
# 2. STYLE
# =========================================================================
def setup_style():
    plt.rcParams.update({
        'font.family':       'Arial',
        'font.size':         10,
        'axes.facecolor':    'white',
        'figure.facecolor':  'white',
        'axes.edgecolor':    '#333333',
        'axes.linewidth':    1.2,
        'axes.grid':         False,
        'xtick.color':       '#333333',
        'ytick.color':       '#333333',
        'text.color':        '#222222',
        'axes.labelcolor':   '#222222',
        'legend.framealpha': 0.9,
        'legend.edgecolor':  '#cccccc',
        'legend.fontsize':   8,
        'pdf.fonttype':      42,
        'ps.fonttype':       42,
    })

setup_style()

PALETTE = [
    '#2166AC','#D6604D','#1A9641','#762A83','#F4A582',
    '#4DAC26','#92C5DE','#E08214','#A6DBA0','#8073AC',
    '#542788','#E66101','#4393C3','#B2182B','#2D7BB6',
    '#FDAE61','#F46D43','#66BD63','#5AAE61','#5E4FA2',
]
CMAP_AUC  = plt.cm.RdYlGn
CMAP_BLUE = LinearSegmentedColormap.from_list(
    'acad_blue', ['#DEEBF7','#9ECAE1','#3182BD','#08519C'])


# =========================================================================
# 3. I/O HELPERS
# =========================================================================
def out_path(name): return os.path.join(CONFIG['out_dir'], name)

def save_fig(fig, stem):
    fig.savefig(out_path(stem + '.pdf'), format='pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(out_path(stem + '.png'), dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  [fig] {stem}.pdf")

def save_csv(df, stem):
    path = out_path(stem + '.csv')
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"  [csv] {stem}.csv")


# =========================================================================
# 4. DATASET
# =========================================================================
class MRIDataset(Dataset):
    def __init__(self, df, transform=None):
        self.data = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, int(row['label']), str(row['patient_id']), row['path']


def get_transforms(train=True):
    """
    Standardised preprocessing consistent with the paper:
      - Extract the 2D axial slice with the maximum tumour area
      - Normalise pixel values to [0, 255]
      - Resize to 224×224 using bilinear interpolation
      - Convert single‑channel grayscale to three‑channel RGB
    The pipeline below reflects these steps after the raw 2D slice
    has been saved as a 3‑channel PNG.
    """
    base = [
        transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]
    if train:
        aug = [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        ]
        return transforms.Compose([base[0]] + aug + base[1:])
    return transforms.Compose(base)


# load_data returns only train and test sets (no validation split)
def load_data():
    train_csv = os.path.join(CONFIG['result_dir'], 'train_dataset.csv')
    test_csv  = os.path.join(CONFIG['result_dir'], 'test_dataset.csv')

    if os.path.exists(train_csv) and os.path.exists(test_csv):
        print("Loading existing train/test split...")
        return pd.read_csv(train_csv), pd.read_csv(test_csv)

    print("Splitting dataset.csv into 80% Train, 20% Test...")
    df = pd.read_csv('dataset.csv')
    train_df, test_df = train_test_split(df, test_size=0.2,
                                         random_state=CONFIG['seed'],
                                         stratify=df['label'])
    train_df.to_csv(train_csv, index=False, encoding='utf-8-sig')
    test_df.to_csv(test_csv,   index=False, encoding='utf-8-sig')
    return train_df, test_df


# =========================================================================
# 5. MODEL REGISTRY
#    Only the 12 models listed above are registered.
#    Removed: AlexNet, MobileNetV3L, SqueezeNet, ShuffleNetV2,
#    SE‑ResNet50, RegNetY‑400MF, ConvNeXt‑Tiny, Swin‑Tiny.
# =========================================================================
class SEBlock(nn.Module):
    """SE block kept for completeness (SE‑ResNet50 has been removed)."""
    def __init__(self, ch, r=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, ch//r, bias=False), nn.ReLU(inplace=True),
            nn.Linear(ch//r, ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)


def build_registry(num_classes):
    """Returns {name: {model, target_layer, paper, params_m}}.
    Only registers the specified 12 architectures.
    """
    R = {}
    def reg(name, model, tl, paper, params_m):
        R[name] = dict(model=model, target_layer=tl, paper=paper, params_m=params_m)

    # VGG16 / VGG19
    for arch, fn, params in [('VGG16', models.vgg16, 138.4),
                              ('VGG19', models.vgg19, 143.7)]:
        m = fn(weights='IMAGENET1K_V1')
        m.classifier[-1] = nn.Sequential(nn.Dropout(0.5), nn.Linear(4096, num_classes))
        reg(arch, m, m.features[-1], 'Simonyan & Zisserman, ICLR 2015', params)

    # ResNet18 / ResNet34 / ResNet50 / ResNet101
    for arch, fn, fc_in, params in [
        ('ResNet18',  models.resnet18,  512,  11.7),
        ('ResNet34',  models.resnet34,  512,  21.8),
        ('ResNet50',  models.resnet50,  2048, 25.6),
        ('ResNet101', models.resnet101, 2048, 44.5),
    ]:
        m = fn(weights='IMAGENET1K_V1')
        mid = fc_in // 2
        m.fc = nn.Sequential(
            nn.Dropout(0.6),
            nn.Linear(fc_in, mid),
            nn.ReLU(inplace=True),
            nn.Dropout(0.6),
            nn.Linear(mid, num_classes)
        )
        reg(arch, m, m.layer4[-1], 'He et al., CVPR 2016', params)

    # DenseNet121 / DenseNet169 / DenseNet201
    for arch, fn, fc_in, params in [
        ('DenseNet121', models.densenet121, 1024, 8.0),
        ('DenseNet169', models.densenet169, 1664, 14.1),
        ('DenseNet201', models.densenet201, 1920, 20.0),
    ]:
        m = fn(weights='IMAGENET1K_V1')
        mid = fc_in // 2
        m.classifier = nn.Sequential(
            nn.Dropout(0.6),
            nn.Linear(fc_in, mid),
            nn.ReLU(inplace=True),
            nn.Dropout(0.6),
            nn.Linear(mid, num_classes)
        )
        reg(arch, m, m.features.denseblock4, 'Huang et al., CVPR 2017', params)

    # MobileNetV2
    m = models.mobilenet_v2(weights='IMAGENET1K_V1')
    m.classifier[-1] = nn.Sequential(nn.Dropout(0.5), nn.Linear(1280, num_classes))
    reg('MobileNetV2', m, m.features[-1], 'Sandler et al., CVPR 2018', 3.4)

    # EfficientNet-B0 / EfficientNet-B4
    m = models.efficientnet_b0(weights='IMAGENET1K_V1')
    m.classifier[-1] = nn.Sequential(nn.Dropout(0.5), nn.Linear(1280, num_classes))
    reg('EfficientNet-B0', m, m.features[-1], 'Tan & Le, ICML 2019', 5.3)

    m = models.efficientnet_b4(weights='IMAGENET1K_V1')
    m.classifier[-1] = nn.Sequential(nn.Dropout(0.7), nn.Linear(1792, num_classes))
    reg('EfficientNet-B4', m, m.features[-1], 'Tan & Le, ICML 2019', 19.3)

    return R


# =========================================================================
# 6. GRAD-CAM++
# =========================================================================
class UniversalGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = self.gradients = None
        self._disable_inplace(model)
        self._register()

    def _disable_inplace(self, module):
        for child in module.modules():
            if hasattr(child, 'inplace'): child.inplace = False

    def _register(self):
        self.target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o))
        self.target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0]))

    def _to_spatial(self, t):
        if t is None: return None
        if t.ndim == 3:
            B, L, C = t.shape; H = W = int(L**0.5)
            t = t.permute(0, 2, 1).reshape(B, C, H, W)
        return t if t.ndim == 4 else None

    def __call__(self, x, class_idx=None, method='gradcam++'):
        self.model.eval()
        self.activations = self.gradients = None
        x = x.to(CONFIG['device']).requires_grad_(True)
        out = self.model(x)
        if class_idx is None: class_idx = out.argmax(1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(out); one_hot[0, class_idx] = 1.
        out.backward(gradient=one_hot, retain_graph=True)

        acts  = self._to_spatial(self.activations)
        grads = self._to_spatial(self.gradients)
        if acts is None or grads is None: return None, class_idx

        if method == 'gradcam++':
            alpha = grads**2 / (2*grads**2 + (grads**3)*acts.sum((2,3), keepdim=True) + 1e-8)
            weights = (alpha * F.relu(grads)).mean((2, 3), keepdim=True)
        else:
            weights = grads.mean((2, 3), keepdim=True)

        cam = F.relu((weights * acts).sum(1, keepdim=True))[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    def overlay(self, cam, raw_rgb, alpha=0.45):
        h, w = raw_rgb.shape[:2]
        cam_up = cv2.resize(cam, (w, h))
        hm = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam_up), cv2.COLORMAP_JET),
                          cv2.COLOR_BGR2RGB)
        return (raw_rgb * (1 - alpha) + hm * alpha).astype(np.uint8), hm


# =========================================================================
# 7. TRAINING
#    train_model now monitors training AUC and saves best weights;
#    the validation loader has been removed.
#    MIXUP_MODELS updated to match the retained architectures.
# =========================================================================
MIXUP_MODELS = {
    'ResNet18', 'ResNet34', 'ResNet50', 'ResNet101',
    'DenseNet121', 'DenseNet169', 'DenseNet201'
}


def train_model(name, model, train_loader, use_mixup=False):
    from sklearn.metrics import roc_auc_score
    device = CONFIG['device']
    model  = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'],
                            weight_decay=CONFIG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
    criterion = nn.CrossEntropyLoss(label_smoothing=0.25)

    weight_path = os.path.join(CONFIG['weights_dir'], f'{name}.pth')
    best_auc, history = 0., defaultdict(list)

    if use_mixup:
        print(f"  [{name}] Mixup enabled (alpha=0.4)")

    for epoch in range(CONFIG['epochs']):
        # Training
        model.train()
        run_loss = correct = total = 0
        t_probs, t_lbls = [], []

        for imgs, labels, _, _ in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()

            if use_mixup:
                lam = np.random.beta(0.4, 0.4)
                idx = torch.randperm(imgs.size(0)).to(device)
                mixed_imgs = lam * imgs + (1 - lam) * imgs[idx]
                out  = model(mixed_imgs)
                loss = lam * criterion(out, labels) + (1 - lam) * criterion(out, labels[idx])
                correct += (out.argmax(1) == labels).sum().item()
            else:
                out  = model(imgs)
                loss = criterion(out, labels)
                correct += (out.argmax(1) == labels).sum().item()

            loss.backward()
            optimizer.step()
            run_loss += loss.item()
            total    += labels.size(0)

            # Collect training probabilities for AUC
            with torch.no_grad():
                t_probs += torch.softmax(out, 1)[:, 1].cpu().tolist()
                t_lbls  += labels.cpu().tolist()

        train_acc = correct / total
        train_auc = roc_auc_score(t_lbls, t_probs) if len(set(t_lbls)) > 1 else 0.5

        history['train_loss'].append(run_loss / len(train_loader))
        history['train_acc'].append(train_acc)
        history['train_auc'].append(train_auc)
        scheduler.step()

        # Save best weights based on training AUC
        if train_auc >= best_auc:
            best_auc = train_auc
            torch.save(model.state_dict(), weight_path)

        print(f"  [{name}] Ep{epoch+1:02d} "
              f"Loss={history['train_loss'][-1]:.4f} "
              f"TrainAcc={train_acc:.3f} "
              f"TrainAUC={train_auc:.3f}"
              + (' ✓' if train_auc >= best_auc else ''))

    return dict(history), weight_path


# =========================================================================
# 8. EVALUATION
# =========================================================================
def _metrics(y, p, pd_):
    if len(set(y)) < 2:
        return dict(acc=0, sen=0, spe=0, pre=0, f1=0, auc=0.5, ap=0,
                    OR=1, OR_lo=0.1, OR_hi=10)
    cm = confusion_matrix(y, pd_)
    tn, fp, fn, tp = (cm.ravel() if cm.size == 4 else [0, 0, 0, 0])
    acc = (tp+tn) / (tp+tn+fp+fn+1e-8)
    sen = tp / (tp+fn+1e-8);  spe = tn / (tn+fp+1e-8)
    pre = tp / (tp+fp+1e-8);  f1  = 2*pre*sen / (pre+sen+1e-8)
    fpr_, tpr_, _ = roc_curve(y, p);  roc_auc = auc(fpr_, tpr_)
    ap = average_precision_score(y, p)
    tp_, fp_, fn_, tn_ = max(tp, 0.5), max(fp, 0.5), max(fn, 0.5), max(tn, 0.5)
    OR = (tp_*tn_) / (fp_*fn_)
    se_log = np.sqrt(1/tp_ + 1/fp_ + 1/fn_ + 1/tn_)
    OR_lo = np.exp(np.log(OR) - 1.96*se_log)
    OR_hi = np.exp(np.log(OR) + 1.96*se_log)
    return dict(acc=acc, sen=sen, spe=spe, pre=pre, f1=f1, auc=roc_auc, ap=ap,
                OR=OR, OR_lo=OR_lo, OR_hi=OR_hi)


def evaluate_model(name, model, loader, weight_path):
    device = CONFIG['device']
    model  = model.to(device)
    if os.path.exists(weight_path):
        model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    model.eval()

    y_all, p_all, pd_all, pid_all = [], [], [], []
    with torch.no_grad():
        for imgs, labels, pids, _ in loader:
            out     = model(imgs.to(device))
            p_all  += torch.softmax(out, 1)[:, 1].cpu().tolist()
            pd_all += out.argmax(1).cpu().tolist()
            y_all  += labels.tolist()
            pid_all += list(pids)

    y = np.array(y_all); p = np.array(p_all); pd_ = np.array(pd_all)
    metrics = _metrics(y, p, pd_)

    ci = defaultdict(list)
    for _ in range(CONFIG['bootstrap_n']):
        idx = np.random.choice(len(y), len(y), replace=True)
        if len(set(y[idx])) < 2: continue
        m_b = _metrics(y[idx], p[idx], pd_[idx])
        for k, v in m_b.items(): ci[k].append(v)
    ci = {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in ci.items()}

    fpr_, tpr_, _ = roc_curve(y, p)
    pre_, rec_, _ = precision_recall_curve(y, p)
    return dict(metrics=metrics, ci=ci,
                labels=y, probs=p, preds=pd_, pids=pid_all,
                fpr=fpr_, tpr=tpr_, precision=pre_, recall=rec_)


# =========================================================================
# 9. GRAD-CAM PANEL
# =========================================================================
def plot_gradcam_panel(name, model, target_layer, loader, weight_path,
                       split='test', n_samples=8):
    device = CONFIG['device']
    model  = model.to(device)
    if os.path.exists(weight_path):
        model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    model.eval()
    cam_gen = UniversalGradCAM(model, target_layer)
    dataset = loader.dataset
    indices = np.random.permutation(len(dataset))[:n_samples*4]

    samples = []
    for idx in indices:
        if len(samples) >= n_samples: break
        img_t, label, pid, img_path = dataset[idx]
        raw_bgr = cv2.imread(img_path)
        if raw_bgr is None: continue
        raw_rgb = cv2.resize(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB),
                             (CONFIG['img_size'], CONFIG['img_size']))
        try:
            cam, pred = cam_gen(img_t.unsqueeze(0), class_idx=1)
        except Exception: continue
        if cam is None: continue
        ov, _ = cam_gen.overlay(cam, raw_rgb)
        with torch.no_grad():
            prob = torch.softmax(model(img_t.unsqueeze(0).to(device)), 1)[0, 1].item()
        samples.append((raw_rgb, cam, ov, label, pred, prob, pid))

    if not samples:
        print(f"  [{name}] No valid samples for Grad-CAM panel"); return []

    n   = len(samples)
    fig = plt.figure(figsize=(16, n*3.5))
    fig.suptitle(f'{name} — Grad-CAM++ Visualisation Panel ({split})',
                 fontsize=13, fontweight='bold', y=1.01)
    gs = gridspec.GridSpec(n, 4, figure=fig, hspace=0.06, wspace=0.04)
    col_titles = ['Original', 'Grad-CAM++ Heatmap', 'Overlay', 'Edge-Enhanced']

    for row_i, (raw, cam, ov, lbl, pred, prob, pid) in enumerate(samples):
        gray   = cv2.cvtColor(raw, cv2.COLOR_RGB2GRAY)
        cam_up = cv2.resize(cam, (raw.shape[1], raw.shape[0]))
        edge   = cv2.normalize((gray * cam_up).astype(np.float32), None, 0, 255,
                               cv2.NORM_MINMAX).astype(np.uint8)
        edge_rgb  = cv2.cvtColor(edge, cv2.COLOR_GRAY2RGB)
        data_cols = [raw, cam, ov, edge_rgb]
        cmaps     = [None, 'YlOrRd', None, None]

        for col_i, (data, cmap) in enumerate(zip(data_cols, cmaps)):
            ax = fig.add_subplot(gs[row_i, col_i])
            if cmap: ax.imshow(data, cmap=cmap, vmin=0, vmax=1)
            else:    ax.imshow(data)
            ax.axis('off')
            if row_i == 0: ax.set_title(col_titles[col_i], fontsize=9, fontweight='bold')
            if col_i == 0:
                cor = '✓' if lbl == pred else '✗'
                col = '#1A9641' if lbl == pred else '#D6604D'
                ax.set_ylabel(f"{cor} {pid[:10]}\nTrue:{CONFIG['class_names'][lbl]}\n"
                              f"Pred:{CONFIG['class_names'][pred]} ({prob:.2f})",
                              color=col, fontsize=7, rotation=0, labelpad=72, va='center')

    plt.tight_layout()
    stem = f"01_gradcam_panel_{name}_{split}"
    save_fig(fig, stem)
    rows = [dict(pid=pid, true=CONFIG['class_names'][lbl],
                 pred=CONFIG['class_names'][pred], prob=f"{prob:.4f}",
                 correct=(lbl == pred))
            for (_, _, _, lbl, pred, prob, pid) in samples]
    save_csv(pd.DataFrame(rows), stem)
    return samples


# =========================================================================
# 10. MULTI-MODEL GRAD-CAM COMPARISON
# =========================================================================
def plot_gradcam_comparison(model_cams, raw_rgb, label, split='test'):
    names = list(model_cams.keys()); n = len(names)
    cols  = 4; rows = (n + cols - 1) // cols
    fig   = plt.figure(figsize=(cols*4.5, rows*4.2+1))
    fig.suptitle(f'Multi-Model Grad-CAM++ Comparison ({split})  '
                 f'| True: {CONFIG["class_names"][label]}',
                 fontsize=13, fontweight='bold')
    gs = gridspec.GridSpec(rows, cols, figure=fig, hspace=0.4, wspace=0.1)

    records = []
    for idx, name in enumerate(names):
        r, c = divmod(idx, cols)
        ax   = fig.add_subplot(gs[r, c])
        cam_arr, ov, pred_idx, prob = model_cams[name]
        ax.imshow(ov)
        correct = pred_idx == label
        bc   = '#1A9641' if correct else '#D6604D'
        mark = '✓' if correct else '✗'
        for sp in ax.spines.values():
            sp.set_edgecolor(bc); sp.set_linewidth(2.5); sp.set_visible(True)
        ax.set_title(f'{name}\n{mark} {CONFIG["class_names"][pred_idx]} ({prob:.3f})',
                     color=bc, fontsize=8, fontweight='bold', pad=3)
        ax.set_xticks([]); ax.set_yticks([])
        ins = ax.inset_axes([0.72, 0., 0.28, 0.28])
        ins.imshow(cam_arr, cmap='YlOrRd', vmin=0, vmax=1); ins.axis('off')
        records.append(dict(model=name, pred=CONFIG['class_names'][pred_idx],
                            prob=f"{prob:.4f}", correct=correct))

    for idx in range(n, rows*cols):
        r, c = divmod(idx, cols)
        fig.add_subplot(gs[r, c]).axis('off')

    h1 = mpatches.Patch(color='#1A9641', label='Correct')
    h2 = mpatches.Patch(color='#D6604D', label='Incorrect')
    fig.legend(handles=[h1, h2], loc='lower center', ncol=2,
               bbox_to_anchor=(0.5, 0.005), fontsize=9)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    stem = f"02_gradcam_comparison_{split}"
    save_fig(fig, stem)
    save_csv(pd.DataFrame(records), stem)


# =========================================================================
# 11. ROC CURVES
# =========================================================================
def plot_roc(train_res, test_res):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    records   = []
    for ax, results, title in zip(axes, [train_res, test_res], ['Training Set', 'Test Set']):
        ax.plot([0, 1], [0, 1], '--', color='#aaaaaa', lw=1.2, label='Random')
        ax.set_xlabel('False Positive Rate', fontsize=11)
        ax.set_ylabel('True Positive Rate',  fontsize=11)
        ax.set_title(f'ROC Curves — {title}', fontsize=12, fontweight='bold')
        sorted_names = sorted(results, key=lambda n: results[n]['metrics']['auc'], reverse=True)
        for i, name in enumerate(sorted_names):
            r       = results[name]; roc_auc = r['metrics']['auc']
            ci      = r['ci'].get('auc', (0, 1))
            col     = PALETTE[i % len(PALETTE)]
            ax.plot(r['fpr'], r['tpr'], color=col, lw=2,
                    label=f"{name}  {roc_auc:.3f} [{ci[0]:.3f}-{ci[1]:.3f}]")
            ax.fill_between(r['fpr'],
                            r['tpr']*ci[0]/(roc_auc+1e-8),
                            r['tpr']*ci[1]/(roc_auc+1e-8),
                            color=col, alpha=0.07)
            records.append(dict(split=title, model=name, AUC=f"{roc_auc:.4f}",
                                CI_lo=f"{ci[0]:.4f}", CI_hi=f"{ci[1]:.4f}"))
        ax.legend(loc='lower right', fontsize=7, framealpha=0.9)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    plt.tight_layout()
    save_fig(fig, "03_roc_comparison")
    save_csv(pd.DataFrame(records), "03_roc_comparison")


# =========================================================================
# 12. PR CURVES
# =========================================================================
def plot_pr(train_res, test_res):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    records   = []
    for ax, results, title in zip(axes, [train_res, test_res], ['Training Set', 'Test Set']):
        ax.set_xlabel('Recall', fontsize=11); ax.set_ylabel('Precision', fontsize=11)
        ax.set_title(f'PR Curves — {title}', fontsize=12, fontweight='bold')
        sorted_names = sorted(results, key=lambda n: results[n]['metrics']['ap'], reverse=True)
        for i, name in enumerate(sorted_names):
            r   = results[name]; ap = r['metrics']['ap']
            col = PALETTE[i % len(PALETTE)]
            ax.plot(r['recall'], r['precision'], color=col, lw=2,
                    label=f"{name}  AP={ap:.3f}")
            records.append(dict(split=title, model=name, AP=f"{ap:.4f}"))
        ax.legend(loc='lower left', fontsize=7)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    plt.tight_layout()
    save_fig(fig, "04_pr_comparison")
    save_csv(pd.DataFrame(records), "04_pr_comparison")


# =========================================================================
# 13. RADAR CHART
# =========================================================================
def plot_radar(train_res, test_res):
    mkeys   = ['acc', 'sen', 'spe', 'pre', 'f1', 'auc', 'ap']
    mlabels = ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1', 'AUC', 'AP']
    N       = len(mkeys)
    angles  = np.linspace(0, 2*np.pi, N, endpoint=False).tolist(); angles += angles[:1]

    fig, axes = plt.subplots(1, 2, figsize=(18, 8), subplot_kw=dict(polar=True))
    records   = []
    for ax, results, title in zip(axes, [train_res, test_res], ['Training', 'Test']):
        ax.set_xticks(angles[:-1]); ax.set_xticklabels(mlabels, fontsize=9)
        ax.set_yticks([.2, .4, .6, .8, 1.])
        ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=7, color='#777777')
        ax.set_title(f'Metric Radar — {title}', fontsize=12, fontweight='bold', pad=15)
        for i, name in enumerate(results):
            vals = [results[name]['metrics'].get(k, 0) for k in mkeys]; vals += vals[:1]
            col  = PALETTE[i % len(PALETTE)]
            ax.plot(angles, vals, color=col, lw=2, label=name)
            ax.fill(angles, vals, color=col, alpha=0.07)
            records.append(dict(split=title, model=name,
                                **{k: f"{results[name]['metrics'].get(k, 0):.4f}" for k in mkeys}))
        ax.legend(fontsize=7, loc='upper right', bbox_to_anchor=(1.35, 1.15))
    plt.tight_layout()
    save_fig(fig, "05_radar")
    save_csv(pd.DataFrame(records), "05_radar")


# =========================================================================
# 14. METRIC BAR CHART + CI
# =========================================================================
def plot_metric_bars(train_res, test_res):
    mkeys   = ['acc', 'sen', 'spe', 'pre', 'f1', 'auc', 'ap']
    mlabels = ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1', 'AUC-ROC', 'AP-PR']
    names   = list(train_res.keys()); n = len(names)
    records = []

    for results, split in [(train_res, 'Train'), (test_res, 'Test')]:
        fig, axes = plt.subplots(2, 4, figsize=(22, 10))
        fig.suptitle(f'Predictive Performance with 95% Bootstrap CI — {split} Set',
                     fontsize=14, fontweight='bold')
        for ax_i, (mk, ml) in enumerate(zip(mkeys, mlabels)):
            ax    = axes.flat[ax_i]
            ax.set_title(ml, fontsize=10, fontweight='bold')
            vals  = [results[nm]['metrics'].get(mk, 0) for nm in names]
            ci_lo = [results[nm]['ci'].get(mk, (0, 0))[0] for nm in names]
            ci_hi = [results[nm]['ci'].get(mk, (0, 0))[1] for nm in names]
            colors = [PALETTE[i % len(PALETTE)] for i in range(n)]
            bars   = ax.bar(range(n), vals, color=colors, alpha=0.85, width=0.7)
            ax.errorbar(range(n), vals,
                        yerr=[np.array(vals)-np.array(ci_lo),
                              np.array(ci_hi)-np.array(vals)],
                        fmt='none', ecolor='#333333', capsize=4, lw=1.5)
            ax.set_xticks(range(n))
            ax.set_xticklabels(names, rotation=45, ha='right', fontsize=7)
            ax.set_ylim(0, 1.12)
            best_i = int(np.argmax(vals))
            bars[best_i].set_edgecolor('#222222'); bars[best_i].set_linewidth(2.5)
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width()/2, v+0.015,
                        f'{v:.3f}', ha='center', va='bottom', fontsize=6)
            for row in zip(names, vals, ci_lo, ci_hi):
                records.append(dict(split=split, metric=ml, model=row[0],
                                    value=f"{row[1]:.4f}", CI_lo=f"{row[2]:.4f}",
                                    CI_hi=f"{row[3]:.4f}"))
        axes.flat[-1].set_visible(False)
        plt.tight_layout()
        save_fig(fig, f"06_metric_bars_{split.lower()}")
    save_csv(pd.DataFrame(records), "06_metric_bars")


# =========================================================================
# 15. CONFUSION MATRIX GRID
# =========================================================================
def plot_cm_grid(train_res, test_res):
    records = []
    for results, split in [(train_res, 'Train'), (test_res, 'Test')]:
        names = list(results.keys()); n = len(names)
        cols  = 4; rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3.8))
        fig.suptitle(f'Confusion Matrices — {split} Set', fontsize=13, fontweight='bold')
        axes = np.array(axes).reshape(-1)
        for i, ax in enumerate(axes):
            if i >= n: ax.axis('off'); continue
            name = names[i]; res = results[name]
            cm   = confusion_matrix(res['labels'], res['preds'])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=CONFIG['class_names'],
                        yticklabels=CONFIG['class_names'],
                        ax=ax, cbar=True, annot_kws={'fontsize': 12})
            ax.set_title(f"{name}\nAUC={res['metrics']['auc']:.3f}", fontsize=9)
            ax.set_xlabel('Predicted', fontsize=8); ax.set_ylabel('True', fontsize=8)
            tn, fp, fn, tp = (cm.ravel() if cm.size == 4 else [0, 0, 0, 0])
            records.append(dict(split=split, model=name, TN=tn, FP=fp, FN=fn, TP=tp))
        plt.tight_layout()
        save_fig(fig, f"07_confusion_matrix_{split.lower()}")
    save_csv(pd.DataFrame(records), "07_confusion_matrix")


# =========================================================================
# 16. SCORE DISTRIBUTION VIOLIN
# =========================================================================
def plot_score_distribution(train_res, test_res):
    records = []
    for results, split in [(train_res, 'Train'), (test_res, 'Test')]:
        names = list(results.keys())
        fig, axes = plt.subplots(1, 2, figsize=(max(18, len(names)*1.5), 7))
        fig.suptitle(f'Prediction Score Distribution — {split} Set',
                     fontsize=13, fontweight='bold')
        for ax_i, (ax, cls) in enumerate(zip(axes, CONFIG['class_names'])):
            ax.set_title(f'{cls} (Label={ax_i}) Sample Scores', fontsize=10, fontweight='bold')
            ax.set_xlabel('Model', fontsize=9); ax.set_ylabel('P(Sensitive)', fontsize=9)
            plot_data, positions = [], []
            for i, name in enumerate(names):
                mask  = results[name]['labels'] == ax_i
                probs = results[name]['probs'][mask]
                if len(probs) > 0:
                    plot_data.append(probs); positions.append(i)
                    for p in probs:
                        records.append(dict(split=split, class_label=cls,
                                            model=name, prob=f"{p:.4f}"))
            parts = ax.violinplot(plot_data, positions=positions,
                                  showmedians=True, showextrema=True)
            for j, pc in enumerate(parts['bodies']):
                pc.set_facecolor(PALETTE[j % len(PALETTE)]); pc.set_alpha(0.65)
            parts['cmedians'].set_color('#222222'); parts['cmedians'].set_linewidth(2)
            for j, (probs, pos) in enumerate(zip(plot_data, positions)):
                jit = np.random.uniform(-.15, .15, size=len(probs))
                ax.scatter(np.full_like(probs, pos) + jit, probs,
                           color=PALETTE[j % len(PALETTE)], alpha=0.35, s=12, zorder=3)
            ax.set_xticks(positions)
            ax.set_xticklabels([names[p] for p in positions],
                               rotation=45, ha='right', fontsize=7.5)
            ax.set_ylim(-.05, 1.05)
            ax.axhline(0.5, color='#D6604D', ls='--', lw=1, alpha=0.7)
        plt.tight_layout()
        save_fig(fig, f"08_score_distribution_{split.lower()}")
    save_csv(pd.DataFrame(records), "08_score_distribution")


# =========================================================================
# 17. CALIBRATION CURVES
# =========================================================================
def plot_calibration(train_res, test_res):
    records   = []
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for ax, results, title in zip(axes, [train_res, test_res], ['Training Set', 'Test Set']):
        ax.plot([0, 1], [0, 1], '--', color='#aaaaaa', lw=1.5, label='Perfect calibration')
        ax.set_xlabel('Mean Predicted Probability', fontsize=11)
        ax.set_ylabel('Fraction of Positives',      fontsize=11)
        ax.set_title(f'Calibration Curve — {title}', fontsize=12, fontweight='bold')
        for i, (name, res) in enumerate(results.items()):
            y, p = res['labels'], res['probs']
            if len(set(y)) < 2: continue
            try:
                frac, mpred = calibration_curve(y, p, n_bins=10)
                ax.plot(mpred, frac, 'o-', color=PALETTE[i % len(PALETTE)],
                        lw=2, ms=5, label=name, alpha=0.9)
                for mp, fp in zip(mpred, frac):
                    records.append(dict(split=title, model=name,
                                        mean_pred=f"{mp:.4f}", frac_pos=f"{fp:.4f}"))
            except Exception: pass
        ax.legend(fontsize=7, loc='upper left')
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    plt.tight_layout()
    save_fig(fig, "09_calibration")
    save_csv(pd.DataFrame(records), "09_calibration")


# =========================================================================
# 18. COMPLEXITY vs AUC BUBBLE
# =========================================================================
def plot_complexity(train_res, test_res, registry):
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    records   = []
    for ax, results, title in zip(axes, [train_res, test_res], ['Training Set', 'Test Set']):
        ax.set_xlabel('Model Parameters (M)', fontsize=11)
        ax.set_ylabel('AUC-ROC',              fontsize=11)
        ax.set_title(f'Complexity vs Performance — {title}\n(bubble size = F1-Score)',
                     fontsize=11, fontweight='bold')
        for i, (name, res) in enumerate(results.items()):
            params = registry.get(name, {}).get('params_m', 10)
            auc_v  = res['metrics']['auc']
            f1_v   = res['metrics']['f1']
            col    = PALETTE[i % len(PALETTE)]
            ax.scatter(params, auc_v, s=f1_v*800+50, color=col, alpha=0.8, zorder=3)
            ax.annotate(name, (params, auc_v), xytext=(5, 4),
                        textcoords='offset points', fontsize=8, fontweight='bold', color=col)
            records.append(dict(split=title, model=name, params_M=params,
                                AUC=f"{auc_v:.4f}", F1=f"{f1_v:.4f}"))
        ax.axhline(0.8, color='#D6604D', ls='--', lw=1, alpha=0.6)
        ax.text(0.5, 0.805, 'AUC=0.80', color='#D6604D', fontsize=8)
    plt.tight_layout()
    save_fig(fig, "10_complexity_auc")
    save_csv(pd.DataFrame(records), "10_complexity_auc")


# =========================================================================
# 19. TRAINING CURVES
# =========================================================================
def plot_training_curves(histories):
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle('Training History Comparison', fontsize=13, fontweight='bold')
    keys   = ['train_loss', 'train_acc', 'train_auc']
    titles = ['Training Loss', 'Training Accuracy', 'Training AUC']
    records = []
    for ax, key, title in zip(axes.flat, keys, titles):
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=9)
        for i, (name, hist) in enumerate(histories.items()):
            vals = hist.get(key, [])
            if vals:
                ax.plot(range(1, len(vals)+1), vals, color=PALETTE[i % len(PALETTE)],
                        lw=2, label=name, alpha=0.9)
                for ep, v in enumerate(vals, 1):
                    records.append(dict(model=name, metric=key, epoch=ep, value=f"{v:.4f}"))
        ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    save_fig(fig, "11_training_curves")
    save_csv(pd.DataFrame(records), "11_training_curves")


# =========================================================================
# 20. GRAD-CAM SSIM MATRIX
# =========================================================================
def plot_ssim_matrix(cam_ref, split='test'):
    if not HAS_SSIM or len(cam_ref) < 2:
        print("  Skipping SSIM matrix (need scikit-image + ≥2 models)"); return
    names   = list(cam_ref.keys()); n = len(names)
    sim_mat = np.eye(n)
    records = []
    for i in range(n):
        for j in range(i+1, n):
            try:    s = ssim(cam_ref[names[i]], cam_ref[names[j]], data_range=1.)
            except: s = 0.
            sim_mat[i, j] = sim_mat[j, i] = s
            records.append(dict(model_A=names[i], model_B=names[j], SSIM=f"{s:.4f}"))

    fig, ax = plt.subplots(figsize=(max(9, n*.9), max(7.5, n*.75)))
    sns.heatmap(sim_mat, annot=True, fmt='.2f', xticklabels=names, yticklabels=names,
                cmap='Blues', vmin=0, vmax=1, ax=ax, annot_kws={'fontsize': 8})
    ax.set_title(f'Grad-CAM Spatial Consistency Matrix (SSIM) — {split}',
                 fontsize=12, fontweight='bold', pad=10)
    ax.tick_params(labelsize=8)
    plt.tight_layout()
    save_fig(fig, f"12_gradcam_ssim_{split}")
    save_csv(pd.DataFrame(records), f"12_gradcam_ssim_{split}")


# =========================================================================
# 21. DeLong TEST HEATMAP
# =========================================================================
def _delong(y, p1, p2):
    def _stats(y, p):
        pos = p[y == 1]; neg = p[y == 0]
        n_p, n_n = len(pos), len(neg)
        if not (n_p and n_n): return 0, 0
        v10 = np.array([np.mean(pi > neg) for pi in pos])
        v01 = np.array([np.mean(pos > ni) for ni in neg])
        return np.mean(v10 > 0.5), np.var(v10, ddof=1)/n_p + np.var(v01, ddof=1)/n_n
    a1, s1 = _stats(y, p1); a2, s2 = _stats(y, p2)
    denom = np.sqrt(s1 + s2 + 1e-12)
    z = (a1 - a2) / denom if denom > 1e-10 else 0.
    return z, 2*(1 - stats.norm.cdf(abs(z)))


def plot_delong(train_res, test_res):
    records = []
    for results, split in [(train_res, 'Train'), (test_res, 'Test')]:
        names = list(results.keys()); n = len(names)
        p_mat = np.ones((n, n)); z_mat = np.zeros((n, n))
        y_ref = results[names[0]]['labels']
        for i in range(n):
            for j in range(i+1, n):
                try:
                    z, p = _delong(y_ref, results[names[i]]['probs'],
                                   results[names[j]]['probs'])
                except: z, p = 0, 1.
                p_mat[i, j] = p_mat[j, i] = p
                z_mat[i, j] = z; z_mat[j, i] = -z
                records.append(dict(split=split, model_A=names[i], model_B=names[j],
                                    z=f"{z:.4f}", p=f"{p:.4f}"))

        fig, axes = plt.subplots(1, 2, figsize=(max(16, n*1.3), max(7, n*.9)))
        fig.suptitle(f'DeLong AUC Significance Test — {split}',
                     fontsize=12, fontweight='bold')
        sns.heatmap(-np.log10(p_mat+1e-10), annot=p_mat, fmt='.3f',
                    xticklabels=names, yticklabels=names, cmap='RdYlGn_r',
                    ax=axes[0], annot_kws={'fontsize': 7})
        axes[0].set_title('-log10(p-value)  [Green=significant]', fontsize=10)
        axes[0].tick_params(labelsize=7)

        vmax = np.abs(z_mat).max() + 0.1
        sns.heatmap(z_mat, annot=True, fmt='.2f',
                    xticklabels=names, yticklabels=names,
                    cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                    ax=axes[1], annot_kws={'fontsize': 7})
        axes[1].set_title('Z-statistic (row vs column)', fontsize=10)
        axes[1].tick_params(labelsize=7)
        plt.tight_layout()
        save_fig(fig, f"13_delong_{split.lower()}")
    save_csv(pd.DataFrame(records), "13_delong")


# =========================================================================
# 22. FOREST PLOT
# =========================================================================
def plot_forest(train_res, test_res):
    def _create_single_forest(res_dict, split_name, color_theme, marker_style):
        names = list(res_dict.keys())
        names = sorted(names, key=lambda n: res_dict[n]['metrics']['OR'], reverse=True)
        n     = len(names)
        aucs  = [res_dict[nm]['metrics']['auc']   for nm in names]
        ORs   = [res_dict[nm]['metrics']['OR']    for nm in names]
        los   = [res_dict[nm]['metrics']['OR_lo'] for nm in names]
        his   = [res_dict[nm]['metrics']['OR_hi'] for nm in names]

        fig = plt.figure(figsize=(12, max(6, n*0.6+2)))
        gs  = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1, 2.5], wspace=0.05)

        ax_left = fig.add_subplot(gs[0])
        norm    = Normalize(vmin=0.5, vmax=1.0)
        cmap_auc = plt.cm.RdYlGn
        y_pos   = np.arange(n)[::-1]
        box_w, box_h = 0.6, 0.75; col_x = 0.5
        for row_i, (name, val) in enumerate(zip(names, aucs)):
            yp    = y_pos[row_i]
            color = cmap_auc(norm(val))
            rect  = plt.Rectangle((col_x - box_w/2, yp - box_h/2), box_w, box_h,
                                   facecolor=color, edgecolor='white', lw=1)
            ax_left.add_patch(rect)
            ax_left.text(col_x, yp, f'{val:.3f}', ha='center', va='center',
                         fontsize=9, fontweight='bold',
                         color='white' if val < 0.75 else '#222222')
        ax_left.set_xlim(0, 1); ax_left.set_ylim(-0.8, n-0.2)
        ax_left.set_yticks(y_pos); ax_left.set_yticklabels(names, fontsize=9)
        ax_left.set_xticks([col_x]); ax_left.set_xticklabels(['AUC'], fontsize=10, fontweight='bold')
        ax_left.xaxis.set_ticks_position('top')
        ax_left.set_title(f'{split_name} Efficacy', fontsize=11, fontweight='bold', pad=20)
        ax_left.grid(False)
        for sp in ax_left.spines.values(): sp.set_visible(False)
        ax_left.axhline(y=-0.5, color='#cccccc', lw=0.8)

        ax_right = fig.add_subplot(gs[1])
        ax_right.axvline(1.0, color='#aaaaaa', lw=1.2, ls='--', zorder=1)
        for row_i, name in enumerate(names):
            yp     = y_pos[row_i]
            or_val = ORs[row_i]; lo_val = los[row_i]; hi_val = his[row_i]
            ax_right.errorbar(or_val, yp,
                              xerr=[[or_val - lo_val], [hi_val - or_val]],
                              fmt=marker_style, color=color_theme, ms=7, lw=2,
                              capsize=5, capthick=1.5, zorder=3)
            ax_right.text(hi_val * 1.1, yp,
                          f'{or_val:.2f} [{lo_val:.2f}-{hi_val:.2f}]',
                          va='center', fontsize=8, color='#333333')
        ax_right.set_xscale('log')
        ax_right.set_xlabel('Odds Ratio (OR) — Log Scale', fontsize=11)
        ax_right.set_title(f'{split_name} OR Forest Plot (Sorted)', fontsize=11, fontweight='bold')
        ax_right.set_ylim(-0.8, n-0.2)
        ax_right.set_yticks(y_pos); ax_right.set_yticklabels([])
        ax_right.axvline(1, color='gray', lw=1, ls='--')

        fig.suptitle(f'Model Performance — {split_name} Set',
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        stem = f"14_forest_plot_{split_name.lower()}"
        save_fig(fig, stem)
        records = [{'Model': nm, 'AUC': f"{a:.4f}", 'OR': f"{o:.4f}",
                    'CI_lo': f"{l:.4f}", 'CI_hi': f"{h:.4f}"}
                   for nm, a, o, l, h in zip(names, aucs, ORs, los, his)]
        save_csv(pd.DataFrame(records), stem)

    _create_single_forest(train_res, 'Train', color_theme='#D6604D', marker_style='^')
    _create_single_forest(test_res,  'Test',  color_theme='#2166AC', marker_style='o')


# =========================================================================
# 23. METRICS SUMMARY TABLE
# =========================================================================
def export_summary(train_res, test_res, registry):
    mkeys = ['acc', 'sen', 'spe', 'pre', 'f1', 'auc', 'ap', 'OR']
    rows  = []
    for name in test_res:
        row = dict(Model=name,
                   Paper=registry.get(name, {}).get('paper', '-'),
                   Params_M=registry.get(name, {}).get('params_m', '-'))
        for split, res in [('Train', train_res), ('Test', test_res)]:
            if name not in res: continue
            for mk in mkeys:
                v  = res[name]['metrics'].get(mk, 0)
                ci = res[name]['ci'].get(mk, (0, 0))
                row[f'{split}_{mk.upper()}'] = f"{v:.4f}"
                if mk != 'OR':
                    row[f'{split}_{mk.upper()}_CI'] = f"[{ci[0]:.4f},{ci[1]:.4f}]"
        rows.append(row)

    df = pd.DataFrame(rows).sort_values('Test_AUC', ascending=False)
    save_csv(df, "15_metrics_summary")

    disp_cols = (['Model', 'Params_M'] +
                 [f'{s}_{m.upper()}' for s in ['Train', 'Test']
                  for m in ['acc', 'sen', 'spe', 'pre', 'f1', 'auc', 'ap']])
    df_d = df[[c for c in disp_cols if c in df.columns]]
    fig, ax = plt.subplots(figsize=(max(22, len(df_d.columns)*1.6), len(df_d)*0.6+2))
    ax.axis('off')
    tbl = ax.table(cellText=df_d.values, colLabels=df_d.columns,
                   loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.9)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#2166AC')
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor('#F7FBFF' if row % 2 == 0 else 'white')
        cell.set_edgecolor('#dddddd')
    fig.suptitle('Multi-Model Performance Summary (sorted by Test AUC)',
                 fontsize=12, fontweight='bold', y=0.98)
    plt.tight_layout()
    save_fig(fig, "15_metrics_summary_table")


# =========================================================================
# Additional: export per‑sample prediction scores (train + test)
# =========================================================================
def export_prediction_scores(train_results, test_results):
    all_records = []
    for results, split in [(train_results, 'Train'), (test_results, 'Test')]:
        split_records = []
        for name, res in results.items():
            y    = res['labels']
            p    = res['probs']
            pd_  = res['preds']
            pids = res['pids']
            for pid, true, pred, prob in zip(pids, y, pd_, p):
                record = dict(
                    model            = name,
                    patient_id       = pid,
                    true_label       = int(true),
                    true_label_name  = CONFIG['class_names'][int(true)],
                    pred_label       = int(pred),
                    pred_label_name  = CONFIG['class_names'][int(pred)],
                    prob_sensitive   = round(float(prob), 6),
                    prob_insensitive = round(1 - float(prob), 6),
                    correct          = int(true) == int(pred),
                    split            = split,
                )
                split_records.append(record)
                all_records.append(record)

        df_split = pd.DataFrame(split_records)
        df_split = df_split.sort_values(['model', 'patient_id']).reset_index(drop=True)
        save_csv(df_split, f"16_prediction_scores_{split.lower()}")

        pivot = df_split.pivot_table(
            index   = ['patient_id', 'true_label', 'true_label_name'],
            columns = 'model',
            values  = 'prob_sensitive',
            aggfunc = 'first'
        ).reset_index()
        pivot.columns.name = None
        save_csv(pivot, f"16_prediction_scores_{split.lower()}_wide")

        print(f"  [{split}] {len(df_split)} records exported "
              f"({df_split['correct'].sum()} correct / {len(df_split)} total predictions)")

    df_all = pd.DataFrame(all_records).sort_values(
        ['split', 'model', 'patient_id']).reset_index(drop=True)
    save_csv(df_all, "16_prediction_scores_all")
    print(f"  [All]  {len(df_all)} total records saved → 16_prediction_scores_all.csv")


# =========================================================================
# 24. MAIN PIPELINE
#    Removed val_df / val_loader; train_model call no longer passes a
#    validation loader.
# =========================================================================
def run():
    print(f"\n{'='*65}")
    print("  Multi-Model Radiomics Framework — Starting")
    print(f"  Device: {CONFIG['device']}  |  Output: {CONFIG['out_dir']}")
    print(f"{'='*65}\n")

    # Data
    train_df, test_df = load_data()

    train_ds = MRIDataset(train_df, get_transforms(train=True))
    test_ds  = MRIDataset(test_df,  get_transforms(train=False))

    train_loader_aug  = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                                   shuffle=True,  num_workers=CONFIG['num_workers'])
    test_loader       = DataLoader(test_ds,  batch_size=1,
                                   shuffle=False, num_workers=CONFIG['num_workers'])
    train_eval_loader = DataLoader(MRIDataset(train_df, get_transforms(False)),
                                   batch_size=1, shuffle=False,
                                   num_workers=CONFIG['num_workers'])

    # Registry
    print("Initialising model registry...")
    registry = build_registry(CONFIG['num_classes'])
    print(f"  {len(registry)} models: {list(registry.keys())}\n")

    histories           = {}
    train_results       = {}
    test_results        = {}
    cam_ref_test        = {}
    all_model_cams_test = {}

    ref_img_t, ref_label, ref_pid, ref_path = test_ds[0]
    ref_raw = cv2.resize(
        cv2.cvtColor(cv2.imread(ref_path), cv2.COLOR_BGR2RGB),
        (CONFIG['img_size'], CONFIG['img_size']))

    # Per-model loop
    for name, info in registry.items():
        print(f"\n{'─'*55}\n  ▶  {name}  ({info['paper']})\n{'─'*55}")
        model        = info['model']
        target_layer = info['target_layer']
        weight_path  = os.path.join(CONFIG['weights_dir'], f'{name}.pth')

        if os.path.exists(weight_path):
            print(f"  [Skip training] weights found: {weight_path}")
            histories[name] = {}
        else:
            t0 = time.time()
            hist, weight_path = train_model(
                name, model, train_loader_aug,
                use_mixup=(name in MIXUP_MODELS)
            )
            histories[name] = hist
            print(f"  Training complete in {time.time()-t0:.0f}s")

        if os.path.exists(weight_path):
            model.load_state_dict(torch.load(weight_path,
                                             map_location=CONFIG['device'],
                                             weights_only=True))

        # Evaluation — TRAIN
        print("  Evaluating on TRAIN set...")
        train_results[name] = evaluate_model(name, model, train_eval_loader, weight_path)
        m = train_results[name]['metrics']
        print(f"  [Train] AUC={m['auc']:.4f}  F1={m['f1']:.4f}  OR={m['OR']:.3f}")

        # Evaluation — TEST
        print("  Evaluating on strictly unseen TEST set...")
        test_results[name] = evaluate_model(name, model, test_loader, weight_path)
        m = test_results[name]['metrics']
        print(f"  [Test]  AUC={m['auc']:.4f}  F1={m['f1']:.4f}  OR={m['OR']:.3f}")

        # Grad-CAM panels
        print("  Generating Grad-CAM panels...")
        plot_gradcam_panel(name, model, target_layer,
                           train_eval_loader, weight_path, 'train', 6)
        plot_gradcam_panel(name, model, target_layer,
                           test_loader,       weight_path, 'test',  6)

        # CAM on reference image
        model_cp = copy.deepcopy(model).to(CONFIG['device'])
        model_cp.load_state_dict(torch.load(weight_path,
                                            map_location=CONFIG['device'],
                                            weights_only=True))
        model_cp.eval()
        cam_gen  = UniversalGradCAM(model_cp, target_layer)
        cam_arr, pred_i = cam_gen(ref_img_t.unsqueeze(0), class_idx=1)
        if cam_arr is not None:
            with torch.no_grad():
                prob = torch.softmax(
                    model_cp(ref_img_t.unsqueeze(0).to(CONFIG['device'])), 1)[0, 1].item()
            ov, _ = cam_gen.overlay(cam_arr, ref_raw)
            all_model_cams_test[name] = (cam_arr, ov, pred_i, prob)
            cam_ref_test[name] = cv2.resize(cam_arr, (CONFIG['img_size'], CONFIG['img_size']))
        del model_cp

    # Summary visualisations
    print(f"\n{'='*55}\n  Generating summary visualisations...\n{'='*55}")

    if all_model_cams_test:
        plot_gradcam_comparison(all_model_cams_test, ref_raw, ref_label, 'test')

    plot_roc(train_results, test_results)
    plot_pr(train_results, test_results)
    plot_radar(train_results, test_results)
    plot_metric_bars(train_results, test_results)
    plot_cm_grid(train_results, test_results)
    plot_score_distribution(train_results, test_results)
    plot_calibration(train_results, test_results)
    plot_complexity(train_results, test_results, registry)

    if any(histories.values()):
        plot_training_curves({k: v for k, v in histories.items() if v})

    if cam_ref_test:
        plot_ssim_matrix(cam_ref_test, 'test')

    plot_delong(train_results, test_results)
    plot_forest(train_results, test_results)
    export_summary(train_results, test_results, registry)
    export_prediction_scores(train_results, test_results)




if __name__ == '__main__':
    run()

